# Task 5 Training Notebook

Use this notebook in Google Colab with only one uploaded dataset file:

- `task5_features.csv`

Expected output folders created by the notebook in the current Colab working directory:

- `models/`
- `output/`
- `figures/`


## Deep Learning Technique Used

This notebook uses a **PyTorch MLP regressor** for supervised tabular regression.

Model type:
- fully connected feedforward neural network
- not an LSTM
- not a CNN
- not a Transformer

Target:
- `target_avg_gas_gwei`

Main input features:
- `cell_tx_count`
- `hour_tx_count`
- `hour_p95_gas_gwei`
- `hour_avg_tx_cost_gwei`
- `hour_total_gas_used`
- `day_tx_count`
- `day_avg_tx_cost_gwei`
- `hour_sin`
- `hour_cos`
- one-hot day-of-week columns

Network architecture:
- `Linear(input_dim, 32)`
- `ReLU`
- `Dropout(0.1)`
- `Linear(32, 16)`
- `ReLU`
- `Linear(16, 1)`

Training setup:
- loss: `MSELoss`
- optimizer: `Adam`
- evaluation: `5-fold` cross-validation
- metrics: `MAE`, `RMSE`, `R^2`
- early stopping with patience `20`


In [ ]:
!pip install -q scikit-learn matplotlib seaborn

# If PyTorch is missing in your Colab runtime, run this once:
# !pip install -q torch


In [ ]:
import json
import random
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


TARGET_COLUMN = 'target_avg_gas_gwei'
IDENTIFIER_COLUMNS = ['hour_of_day', 'day_of_week']
DEFAULT_SEED = 42


def find_data_path() -> Path:
    candidates = [
        Path.cwd() / 'task5_features.csv',
        Path('/content/task5_features.csv'),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    matches = sorted(Path('/content').rglob('task5_features.csv'))
    if matches:
        return matches[0]

    raise FileNotFoundError(
        'Could not find task5_features.csv. Upload it to Colab before running the notebook.'
    )


PROJECT_ROOT = Path.cwd()
DATA_PATH = find_data_path()
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR = PROJECT_ROOT / 'output'
FIGURES_DIR = PROJECT_ROOT / 'figures'
MODEL_PATH = MODELS_DIR / 'task5_mlp.pt'
PREDICTIONS_PATH = OUTPUT_DIR / 'task5_predictions.csv'
METRICS_PATH = OUTPUT_DIR / 'task5_metrics.json'
LOSS_PLOT_PATH = FIGURES_DIR / 'task5_loss_curve.png'
PRED_PLOT_PATH = FIGURES_DIR / 'task5_pred_vs_actual.png'
RESIDUAL_PLOT_PATH = FIGURES_DIR / 'task5_residuals.png'

for directory in (MODELS_DIR, OUTPUT_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_PATH: {DATA_PATH}')
print(f'MODELS_DIR: {MODELS_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'FIGURES_DIR: {FIGURES_DIR}')


In [ ]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    df = pd.read_csv(data_path)
    required = {
        'hour_of_day',
        'day_of_week',
        TARGET_COLUMN,
        'cell_tx_count',
        'hour_tx_count',
        'hour_p95_gas_gwei',
        'hour_avg_tx_cost_gwei',
        'hour_total_gas_used',
        'day_tx_count',
        'day_avg_tx_cost_gwei',
        'hour_sin',
        'hour_cos',
    }
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f'Dataset is missing required columns: {missing}')
    return df


def feature_columns(df: pd.DataFrame) -> list[str]:
    excluded = set(IDENTIFIER_COLUMNS + [TARGET_COLUMN])
    return [column for column in df.columns if column not in excluded]


df = load_dataset(DATA_PATH)
features = feature_columns(df)
print(f'Dataset shape: {df.shape}')
print('Feature columns:')
for column in features:
    print(f' - {column}')

df.head()


In [ ]:
def set_seed(seed: int = DEFAULT_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class MLPRegressor(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.layers(features)


def regression_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    return {
        'mae': float(mean_absolute_error(actual, predicted)),
        'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
        'r2': float(r2_score(actual, predicted)),
    }


def build_dataloaders(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    batch_size: int,
):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    train_dataset = TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32).unsqueeze(1),
    )
    val_dataset = TensorDataset(
        torch.tensor(X_val_scaled, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32).unsqueeze(1),
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, scaler


def train_one_fold(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    *,
    device: torch.device,
    input_dim: int,
    batch_size: int = 16,
    learning_rate: float = 1e-3,
    epochs: int = 250,
    patience: int = 20,
) -> dict[str, Any]:
    train_loader, val_loader, scaler = build_dataloaders(
        X_train, y_train, X_val, y_val, batch_size
    )

    model = MLPRegressor(input_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}

    for _epoch in range(epochs):
        model.train()
        train_loss_total = 0.0
        for features_batch, target_batch in train_loader:
            features_batch = features_batch.to(device)
            target_batch = target_batch.to(device)

            optimizer.zero_grad()
            prediction = model(features_batch)
            loss = criterion(prediction, target_batch)
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * len(features_batch)

        train_loss = train_loss_total / len(train_loader.dataset)

        model.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for features_batch, target_batch in val_loader:
                features_batch = features_batch.to(device)
                target_batch = target_batch.to(device)
                prediction = model(features_batch)
                loss = criterion(prediction, target_batch)
                val_loss_total += loss.item() * len(features_batch)

        val_loss = val_loss_total / len(val_loader.dataset)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss - 1e-8:
            best_val_loss = val_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is None:
        raise RuntimeError('Training failed to produce a model state.')

    model.load_state_dict(best_state)
    X_val_scaled = scaler.transform(X_val)
    with torch.no_grad():
        val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
        predicted = model(val_tensor).cpu().numpy().reshape(-1)

    return {
        'history': history,
        'metrics': regression_metrics(y_val, predicted),
        'predicted': predicted,
        'model_state_dict': best_state,
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist(),
    }


def save_loss_plot(history: dict[str, list[float]], path: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Task 5 Training Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def save_pred_plot(actual: np.ndarray, predicted: np.ndarray, path: Path) -> None:
    min_value = float(min(actual.min(), predicted.min()))
    max_value = float(max(actual.max(), predicted.max()))

    plt.figure(figsize=(6, 6))
    plt.scatter(actual, predicted, alpha=0.75)
    plt.plot([min_value, max_value], [min_value, max_value], linestyle='--')
    plt.xlabel('Actual Avg Gas Gwei')
    plt.ylabel('Predicted Avg Gas Gwei')
    plt.title('Actual vs Predicted Gas Fee')
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def save_residual_plot(actual: np.ndarray, predicted: np.ndarray, path: Path) -> None:
    residuals = actual - predicted
    plt.figure(figsize=(8, 5))
    plt.scatter(predicted, residuals, alpha=0.75)
    plt.axhline(0.0, linestyle='--')
    plt.xlabel('Predicted Avg Gas Gwei')
    plt.ylabel('Residual')
    plt.title('Prediction Residuals')
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


In [ ]:
def run_training_pipeline(
    df: pd.DataFrame,
    *,
    seed: int = DEFAULT_SEED,
    batch_size: int = 16,
    learning_rate: float = 1e-3,
    epochs: int = 250,
    patience: int = 20,
) -> dict[str, Any]:
    features = feature_columns(df)
    identifiers = df[IDENTIFIER_COLUMNS].copy()

    X = df[features].to_numpy(dtype=np.float32)
    y = df[TARGET_COLUMN].to_numpy(dtype=np.float32)

    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    splitter = KFold(n_splits=5, shuffle=True, random_state=seed)

    oof_predictions = np.zeros(len(df), dtype=np.float32)
    fold_metrics = []
    best_fold_history = None
    best_checkpoint = None
    best_rmse = float('inf')

    for fold_number, (train_index, val_index) in enumerate(splitter.split(X), start=1):
        result = train_one_fold(
            X[train_index],
            y[train_index],
            X[val_index],
            y[val_index],
            device=device,
            input_dim=len(features),
            batch_size=batch_size,
            learning_rate=learning_rate,
            epochs=epochs,
            patience=patience,
        )

        oof_predictions[val_index] = result['predicted']
        fold_summary = {'fold': fold_number, **result['metrics']}
        fold_metrics.append(fold_summary)

        if result['metrics']['rmse'] < best_rmse:
            best_rmse = result['metrics']['rmse']
            best_fold_history = result['history']
            best_checkpoint = {
                'model_state_dict': result['model_state_dict'],
                'scaler_mean': result['scaler_mean'],
                'scaler_scale': result['scaler_scale'],
                'feature_columns': features,
                'target_column': TARGET_COLUMN,
                'seed': seed,
                'fold': fold_number,
            }

    if best_fold_history is None or best_checkpoint is None:
        raise RuntimeError('Cross-validation did not produce a checkpoint.')

    overall_metrics = regression_metrics(y, oof_predictions)

    predictions_df = identifiers.copy()
    predictions_df['actual_avg_gas_gwei'] = y
    predictions_df['predicted_avg_gas_gwei'] = oof_predictions
    predictions_df['absolute_error'] = np.abs(y - oof_predictions)
    predictions_df.to_csv(PREDICTIONS_PATH, index=False)

    metrics_payload = {
        'dataset_rows': int(len(df)),
        'device': str(device),
        'feature_columns': features,
        'target_column': TARGET_COLUMN,
        'cross_validation_folds': 5,
        'fold_metrics': fold_metrics,
        'overall_metrics': overall_metrics,
        'best_checkpoint_fold': int(best_checkpoint['fold']),
    }
    with METRICS_PATH.open('w', encoding='utf-8') as handle:
        json.dump(metrics_payload, handle, indent=2)

    torch.save(best_checkpoint, MODEL_PATH)
    save_loss_plot(best_fold_history, LOSS_PLOT_PATH)
    save_pred_plot(y, oof_predictions, PRED_PLOT_PATH)
    save_residual_plot(y, oof_predictions, RESIDUAL_PLOT_PATH)

    return {
        'metrics': metrics_payload,
        'predictions_preview': predictions_df.head(10),
        'saved_files': {
            'model_path': str(MODEL_PATH),
            'predictions_path': str(PREDICTIONS_PATH),
            'metrics_path': str(METRICS_PATH),
            'loss_plot_path': str(LOSS_PLOT_PATH),
            'pred_plot_path': str(PRED_PLOT_PATH),
            'residual_plot_path': str(RESIDUAL_PLOT_PATH),
        },
    }


results = run_training_pipeline(df)
results['predictions_preview']


## Result Summary From Generated Outputs

Based on the current generated files in `deep-learning/output` and `deep-learning/figures`, the model result is:

- dataset rows: `168`
- device used: `cpu`
- cross-validation folds: `5`
- overall `MAE`: `0.2664`
- overall `RMSE`: `0.4259`
- overall `R^2`: `0.7497`
- best checkpoint fold: `4`

Fold-level results:
- fold 1: `MAE 0.3271`, `RMSE 0.5663`, `R^2 0.5249`
- fold 2: `MAE 0.3174`, `RMSE 0.5262`, `R^2 0.7604`
- fold 3: `MAE 0.2458`, `RMSE 0.3415`, `R^2 0.8498`
- fold 4: `MAE 0.1801`, `RMSE 0.2232`, `R^2 0.8341`
- fold 5: `MAE 0.2588`, `RMSE 0.3708`, `R^2 0.7793`

Interpretation of the figures:
- `task5_loss_curve.png`: training and validation loss both drop quickly and then stabilize, so the model converges without obvious instability.
- `task5_pred_vs_actual.png`: most points follow the diagonal trend, which means the model captures the general gas-fee pattern reasonably well.
- the scatter widens at higher gas-fee values, so the model is less accurate on more extreme congestion cases.
- `task5_residuals.png`: residuals are centered around zero overall, but larger positive and negative errors appear in the higher predicted range.

Plain-language metric explanation:
- this is **not** `7% loss`.
- `MAE = 0.2664` means the prediction is off by about `0.27 gwei` on average.
- `RMSE = 0.4259` means larger mistakes exist, and this metric punishes those bigger misses more strongly than MAE.
- `R^2 = 0.7497` means the model explains about `75%` of the variation in gas fees, but it does **not** mean `75% accuracy`.
- in simple terms: the model works reasonably well overall, but it becomes less accurate on the more extreme gas-fee spikes.

Conclusion:
- the MLP is a reasonable deep learning baseline for this aggregated congestion dataset.
- it explains about `75%` of the target variance (`R^2 ~ 0.75`).
- it works better on normal gas-fee periods than on the most extreme spikes.


In [ ]:
print(json.dumps(results['metrics']['overall_metrics'], indent=2))
print('\nSaved files:')
for name, path in results['saved_files'].items():
    print(f' - {name}: {path}')
